# Lab C: Topic modeling on book titles

**Decision log**

| Choice | What we did | Why |
|---|---|---|
| Duplicates | removed (142) | repeated titles count twice |
| Train/test split | none | no labels, so nothing to test |
| Cleaning | lowercase (vectorizer default) | keeps words that differ only by case together |
| Stop words | sklearn list + noise list | words like edition, card, package are not subjects |
| min_df | 5 | rarer words show no pattern |
| LDA features | CountVectorizer | LDA works on counts |
| NMF features | TfidfVectorizer | lowers weight of common words |
| random_state | 2 | same results every run |
| Number of topics | 10 | clearest topics (see step 6) |

## Step 1: Explore

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation, NMF

random_state = 2
np.random.seed(random_state)

with open("data/all_book_titles.txt", encoding="utf-8") as f:
    titles = [t.strip() for t in f.read().splitlines() if t.strip()]

df = pd.DataFrame({"title": titles})
print("titles:", len(df))
print("duplicates:", len(df) - df["title"].nunique())

df["n_words"] = df["title"].str.split().str.len()
print(df["n_words"].describe())

print(df["title"].str.lower().str.split().explode().value_counts().head(40))

titles: 2373
duplicates: 142
count    2373.000000
mean        7.142857
std         4.096361
min         1.000000
25%         4.000000
50%         6.000000
75%         9.000000
max        36.000000
Name: n_words, dtype: float64
title
and             932
of              576
the             489
edition)        321
to              320
a               304
in              304
for             258
introduction    236
with            180
an              175
edition         136
:               130
history         104
&               104
science         100
computer         88
approach         87
access           77
business         75
human            71
applications     68
principles       67
analysis         64
essentials       63
series           62
biology          61
art              58
theory           58
guide            56
fundamentals     56
statistics       53
card             53
chemistry        52
concepts         51
anatomy          51
design           50
data             50
world  

There are 2373 titles and 142 are duplicates. The median title is 6 words long. The most common words are stop words plus publisher words like edition, introduction, access, series and card.

Noise words: edition, th, s, access, introduction, approach, 5th, volume, series, book, brief, essentials, card, real.

## Step 2: Clean and vectorize

In [2]:
df = df.drop_duplicates(subset="title").reset_index(drop=True)
print("titles left:", len(df))

noise = ["edition", "th", "s", "access", "introduction", "approach", "5th",
         "volume", "series", "book", "brief", "essentials", "card", "real"]
stop_list = list(ENGLISH_STOP_WORDS) + noise

count_vec = CountVectorizer(stop_words=stop_list, min_df=5)
X_counts = count_vec.fit_transform(df["title"])

tfidf_vec = TfidfVectorizer(stop_words=stop_list, min_df=5)
X_tfidf = tfidf_vec.fit_transform(df["title"])

print(X_counts.shape, X_tfidf.shape)

titles left: 2231
(2231, 437) (2231, 437)


LDA gets counts because it is a probability model built on how many times words show up. NMF can use tf-idf because it is just a matrix factorization.

## Step 3: Fit both models (k = 10, chosen in step 6)

In [3]:
K = 10
lda = LatentDirichletAllocation(n_components=K, random_state=random_state)
lda.fit(X_counts)

nmf = NMF(n_components=K, random_state=random_state)
nmf.fit(X_tfidf)

,"n_components n_components: int or {'auto'} or None, default='auto'Number of components. If `None`, all features are kept.If `n_components='auto'`, the number of components is automatically inferredfrom W or H shapes... versionchanged:: 1.4 Added `'auto'` value... versionchanged:: 1.6 Default value changed from `None` to `'auto'`.",10
,"random_state random_state: int, RandomState instance or None, default=NoneUsed for initialisation (when ``init`` == 'nndsvdar' or'random'), and in Coordinate Descent. Pass an int for reproducibleresults across multiple function calls.See :term:`Glossary <random_state>`.",2
,"init init: {'random', 'nndsvd', 'nndsvda', 'nndsvdar', 'custom'}, default=NoneMethod used to initialize the procedure.Valid options:- `None`: 'nndsvda' if n_components <= min(n_samples, n_features), otherwise random.- `'random'`: non-negative random matrices, scaled with: `sqrt(X.mean() / n_components)`- `'nndsvd'`: Nonnegative Double Singular Value Decomposition (NNDSVD) initialization (better for sparseness)- `'nndsvda'`: NNDSVD with zeros filled with the average of X (better when sparsity is not desired)- `'nndsvdar'` NNDSVD with zeros filled with small random values (generally faster, less accurate alternative to NNDSVDa for when sparsity is not desired)- `'custom'`: Use custom matrices `W` and `H` which must both be provided... versionchanged:: 1.1 When `init=None` and n_components is less than n_samples and n_features defaults to `nndsvda` instead of `nndsvd`.",None
,"solver solver: {'cd', 'mu'}, default='cd'Numerical solver to use:- 'cd' is a Coordinate Descent solver.- 'mu' is a Multiplicative Update solver... versionadded:: 0.17 Coordinate Descent solver... versionadded:: 0.19 Multiplicative Update solver.",'cd'
,"beta_loss beta_loss: float or {'frobenius', 'kullback-leibler', 'itakura-saito'}, default='frobenius'Beta divergence to be minimized, measuring the distance between Xand the dot product WH. Note that values different from 'frobenius'(or 2) and 'kullback-leibler' (or 1) lead to significantly slowerfits. Note that for beta_loss <= 0 (or 'itakura-saito'), the inputmatrix X cannot contain zeros. Used only in 'mu' solver... versionadded:: 0.19",'frobenius'
,"tol tol: float, default=1e-4Tolerance of the stopping condition.",0.0001
,"max_iter max_iter: int, default=200Maximum number of iterations before timing out.",200
,"alpha_W alpha_W: float, default=0.0Constant that multiplies the regularization terms of `W`. Set it to zero(default) to have no regularization on `W`... versionadded:: 1.0",0.0
,"alpha_H alpha_H: float or ""same"", default=""same""Constant that multiplies the regularization terms of `H`. Set it to zero tohave no regularization on `H`. If ""same"" (default), it takes the same value as`alpha_W`... versionadded:: 1.0",'same'
,"l1_ratio l1_ratio: float, default=0.0The regularization mixing parameter, with 0 <= l1_ratio <= 1.For l1_ratio = 0 the penalty is an elementwise L2 penalty(aka Frobenius Norm).For l1_ratio = 1 it is an elementwise L1 penalty.For 0 < l1_ratio < 1, the penalty is a combination of L1 and L2... versionadded:: 0.17 Regularization parameter *l1_ratio* used in the Coordinate Descent solver.",0.0
,"verbose verbose: int, default=0Whether to be verbose.",0


## Step 4: Top 10 words

In [4]:
def show_topics(model, vectorizer, n=10):
    words = vectorizer.get_feature_names_out()
    for i, comp in enumerate(model.components_):
        top = [words[j] for j in comp.argsort()[-n:][::-1]]
        print("Topic", i, ":", ", ".join(top))

print("LDA")
show_topics(lda, count_vec)
print()
print("NMF")
show_topics(nmf, tfidf_vec)

LDA
Topic 0 : analysis, data, design, structures, security, pharmacology, nutrition, health, java, using
Topic 1 : marketing, science, learning, 3rd, political, machine, computer, business, feminism, perspective
Topic 2 : package, plus, earth, etext, sociology, early, 8th, forestry, science, relativity
Topic 3 : biology, world, applications, psychology, molecular, guide, understanding, life, concepts, pathophysiology
Topic 4 : economics, finance, human, concepts, insurance, sexuality, hill, mcgraw, latin, america
Topic 5 : history, theory, art, american, film, modern, sex, game, gender, practice
Topic 6 : anthropology, philosophy, buddhism, judaism, differential, equations, hinduism, problems, 2nd, accounting
Topic 7 : computer, systems, science, management, design, database, organization, programming, business, operating
Topic 8 : statistics, probability, language, anatomy, physiology, manual, natural, processing, engineers, student
Topic 9 : chemistry, mechanics, engineering, organic

In [5]:
# labels (made after reading the top titles in step 5)
nmf_labels = {
    0: "History, art and film",
    1: "Computer science (earth and political science mixed in)",
    2: "Biology",
    3: "Business and statistics",
    4: "Human anatomy and physiology",
    5: "Chemistry",
    6: "Analysis, design and data systems (mixed)",
    7: "Unclear",
    8: "Sociology and world religions",
    9: "Psychology",
}

lda_labels = {
    0: "Unclear",
    1: "Machine learning (one political science title mixed in)",
    2: "Unclear (publisher filler: package, plus, etext)",
    3: "Unclear",
    4: "Corporate finance",
    5: "Art history, theater and performance",
    6: "Unclear",
    7: "Unclear",
    8: "Natural language processing",
    9: "Unclear",
}

NMF topic 7 is unclear: the top titles are all game theory, but the top words (theory, game, practice, probability, film, relativity, actuarial) don't say one subject. NMF topic 1 is computer science but also has earth and political science words. NMF topic 4 is anatomy and physiology but publisher words (plus, masteringa, package, etext) leaked into it.

LDA has 6 unclear topics: 0 (java, security, pharmacology, internet law), 2 (just publisher filler: package, plus, etext), 3 (pathophysiology and microprocessors), 6 (religion, anthropology and differential equations), 7 (a mix of unrelated books), 9 (chemistry and literature anthologies).

## Step 5: Top 5 titles per topic

In [6]:
nmf_scores = nmf.transform(X_tfidf)
lda_scores = lda.transform(X_counts)

def show_top_titles(scores, labels, name):
    for topic in range(scores.shape[1]):
        print(name, "topic", topic, "-", labels[topic])
        for pos in scores[:, topic].argsort()[-5:][::-1]:
            print("   ", df["title"].iloc[pos][:95])
        print()

show_top_titles(nmf_scores, nmf_labels, "NMF")
show_top_titles(lda_scores, lda_labels, "LDA")

NMF topic 0 - History, art and film
    Art : A Brief History
    Art History Volume 1 (5th Edition)
    Art: A Brief History
    Art History Portables Book 6 (5th Edition)
    Art History Portable Book 1 (5th Edition)

NMF topic 1 - Computer science (earth and political science mixed in)
    Connecting with Computer Science
    Computer Science Illuminated
    Invitation to Computer Science
    Computer Science : An Overview
    Art & Science of Java An Introduction to Computer Science

NMF topic 2 - Biology
    Bscs Biology A Molecular Approach
    Molecular Biology of the Gene
    Molecular Biology
    Molecular Biology: Genes to Proteins
    Molecular Biology of Cancer: Mechanisms, Targets, and Therapeutics

NMF topic 3 - Business and statistics
    Statistics for Business and Economics (8th Edition)
    Business Statistics: Communicating with Numbers
    Essentials of Business Statistics
    Applied Statistics in Business and Economics
    Statistics for Business and Economics (wi

NMF titles matched our labels for 9 of the 10 topics (art history, computer science, molecular biology, business statistics, anatomy and physiology, chemistry, sociology, psychology, and game theory for the unclear one). Topic 6 is real analysis and regression, which fits analysis but not design or data systems. LDA matched for 4 topics (machine learning, corporate finance, natural language processing, and art and theater in topic 5). The rest were mixed. LDA works from raw counts, so groups of near identical titles with long repeated series names (the NLP conference proceedings, the Gardner art books, Printed Access Card titles) pull a whole topic toward them. Many NMF titles tie on score (all the Art History editions look alike), so the five shown can change a bit.

## Step 6: Number of topics

In [7]:
def top_words(model, vectorizer, n=8):
    words = vectorizer.get_feature_names_out()
    return [", ".join(words[j] for j in comp.argsort()[-n:][::-1]) for comp in model.components_]

results = []
for k in [5, 10, 15]:
    lda_k = LatentDirichletAllocation(n_components=k, random_state=random_state).fit(X_counts)
    nmf_k = NMF(n_components=k, random_state=random_state).fit(X_tfidf)
    results.append({"k": k, "NMF error": nmf_k.reconstruction_err_})
    print("k =", k)
    print("LDA")
    for i, w in enumerate(top_words(lda_k, count_vec)):
        print("  ", i, w)
    print("NMF")
    for i, w in enumerate(top_words(nmf_k, tfidf_vec)):
        print("  ", i, w)
    print()

pd.DataFrame(results).round(2)

k = 5
LDA
   0 mechanics, data, probability, statistics, health, principles, structures, engineers
   1 business, computer, management, economics, sociology, systems, physics, learning
   2 history, art, physiology, anatomy, plus, package, literature, american
   3 science, theory, biology, applications, psychology, analysis, guide, design
   4 language, finance, chemistry, concepts, anthropology, student, human, manual
NMF
   0 history, art, american, world, america, film, modern, global
   1 science, computer, earth, political, organization, networks, physical, architecture
   2 biology, molecular, chemistry, concepts, life, campbell, biochemistry, cell
   3 business, statistics, economics, theory, analysis, probability, applications, principles
   4 human, physiology, anatomy, sexuality, plus, anthropology, masteringa, package



k = 10
LDA
   0 analysis, data, design, structures, security, pharmacology, nutrition, health
   1 marketing, science, learning, 3rd, political, machine, computer, business
   2 package, plus, earth, etext, sociology, early, 8th, forestry
   3 biology, world, applications, psychology, molecular, guide, understanding, life
   4 economics, finance, human, concepts, insurance, sexuality, hill, mcgraw
   5 history, theory, art, american, film, modern, sex, game
   6 anthropology, philosophy, buddhism, judaism, differential, equations, hinduism, problems
   7 computer, systems, science, management, design, database, organization, programming
   8 statistics, probability, language, anatomy, physiology, manual, natural, processing
   9 chemistry, mechanics, engineering, organic, physical, music, literature, thermodynamics
NMF
   0 history, art, american, world, america, film, modern, myartslab
   1 science, computer, earth, political, organization, networks, architecture, physical
   2 biolog

k = 15
LDA
   0 data, fundamentals, structures, security, pharmacology, java, equations, differential
   1 marketing, american, 3rd, sex, microbiology, feminism, gender, women
   2 applications, buddhism, physics, relativity, forestry, basic, theater, 6th
   3 biology, theory, molecular, world, guide, life, game, practice
   4 economics, finance, concepts, insurance, east, analysis, hill, mcgraw
   5 history, art, applied, modern, 2nd, film, american, course
   6 anthropology, contemporary, clinical, diaspora, actuarial, mathematics, immunology, medicine
   7 computer, systems, science, design, database, organization, global, law
   8 probability, language, manual, engineers, processing, natural, statistics, laboratory
   9 engineering, principles, mechanics, philosophy, judaism, quantum, thermodynamics, greek
   10 chemistry, calculus, human, sexuality, early, 4th, organic, forensic
   11 statistics, student, handbook, 9th, latin, america, criminology, business
   12 nutrition, busine

,k,NMF error
0,5,45.53
1,10,44.35
2,15,43.27


At k = 5 the topics were still mixed, and one LDA topic was mostly publisher filler like plus and package. At k = 10 the topics were clearer, with separate finance, religion, computing and biology topics. At k = 15 some topics started to overlap, like the systems and management ones.

NMF error went from 45.53 (k = 5) to 44.35 (k = 10) to 43.27 (k = 15). It keeps going down as k grows because more topics always fit the data better, so it can't pick k for us. We did not use LDA perplexity. We picked k = 10 because the topics were the clearest to read.

## Step 7: Topic mixtures for five titles

In [8]:
def show_mixture(pos, scores, labels, name):
    row = scores[pos] / (scores[pos].sum() + 1e-9)
    order = row.argsort()[::-1][:3]
    print("   ", name, ":", "; ".join(f"T{t} {labels[t]} {row[t]:.0%}" for t in order))

for pos in [0, 100, 500, 1000, 2000]:
    print(df["title"].iloc[pos][:110])
    show_mixture(pos, nmf_scores, nmf_labels, "NMF")
    show_mixture(pos, lda_scores, lda_labels, "LDA")
    print()

Philosophy of Sex and Love A Reader
    NMF : T8 Sociology and world religions 44%; T0 History, art and film 20%; T9 Psychology 11%
    LDA : T6 Unclear 50%; T5 Art history, theater and performance 30%; T9 Unclear 3%

The Theater Experience
    NMF : T0 History, art and film 36%; T9 Psychology 26%; T8 Sociology and world religions 22%
    LDA : T2 Unclear (publisher filler: package, plus, etext) 70%; T5 Art history, theater and performance 3%; T9 Unclear 3%

The Convergence of Race, Ethnicity, and Gender: Multiple Identities in Counseling (4th Edition) (Merrill Couns
    NMF : T8 Sociology and world religions 36%; T9 Psychology 16%; T0 History, art and film 15%
    LDA : T5 Art history, theater and performance 82%; T3 Unclear 2%; T2 Unclear (publisher filler: package, plus, etext) 2%

Essentials of Human Anatomy & Physiology (10th Edition)
    NMF : T4 Human anatomy and physiology 94%; T9 Psychology 6%; T7 Unclear 0%
    LDA : T8 Natural language processing 82%; T3 Unclear 2%; T2 Uncle

Essentials of Human Anatomy & Physiology is about one thing: NMF puts 94% on the anatomy topic. The Theater Experience is spread over three topics (36%, 26%, 22%). Philosophy of Sex and Love is mostly sociology (44%) with some history and art (20%). The Convergence of Race, Ethnicity, and Gender is 36% sociology and then spread thin. Essentials of Pharmacology is 49% on psychology, probably because "health" is a top word there and there is no pharmacology topic at k = 10. Titles with specific technical words land on one topic, and broad or short titles spread out.

## Investigations

(to add)